In [ ]:
# %% [markdown]
# # Part 3: NLP and Sequence Modeling Mini Project
# 
# This notebook builds an end-to-end NLP pipeline for customer support sentiment classification, 
# comparing a traditional TF-IDF + Logistic Regression baseline against an LSTM sequence model.
# Running this notebook will automatically generate your expected repository deliverables.

# %%
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Setup output folders
os.makedirs('results', exist_ok=True)

# Fetch essential NLP tools
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# %% [markdown]
# ## Task 1: Dataset Understanding
# We ingest the `customer_support_text_classification.csv` file, analyze its shapes, and log properties.

# %%
df = pd.read_csv('customer_support_text_classification.csv')

print("--- Task 1: Dataset Insights ---")
print(f"Number of records: {len(df)}")
print(f"Target classes: {df['sentiment_label'].unique()}")
print(f"Average text length: {df['customer_message'].apply(lambda x: len(str(x).split())).mean():.2f} words\n")
print("Class Distribution:")
print(df['sentiment_label'].value_counts())
print("\nSample Data Structure:")
display(df[['customer_message', 'sentiment_label']].head(3))

# %% [markdown]
# ## Task 2: Text Preprocessing
# Standardizing customer text by applying lowercasing, stripping special signs, tokenizing, and filtering stop words.

# %%
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-str0-9\s]', '', text)
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words]
    return " ".join(filtered)

df['clean_message'] = df['customer_message'].apply(preprocess_text)
print("Text cleaning step executed successfully.")

# %% [markdown]
# ## Task 3: Text Vectorization
# Mapping target strings to discrete integers and creating numeric feature footprints via both TF-IDF matrix pools and integer paddings.

# %%
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment_label'].map(label_mapping)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_message'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# 1. TF-IDF Vectors for Baseline
tfidf_vectorizer = TfidfVectorizer(max_features=3000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)
X_test_tfidf = tfidf_vectorizer.transform(X_test_raw)

# 2. Tokenizer Sequence Vectors for Deep Learning
max_words = 3000
max_len = 40
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_raw)

X_train_seq = tokenizer.texts_to_sequences(X_train_raw)
X_test_seq = tokenizer.texts_to_sequences(X_test_raw)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

print(f"TF-IDF Feature Space Dimensions: {X_train_tfidf.shape}")
print(f"Padded Sequence Matrix Dimensions: {X_train_pad.shape}")

# %% [markdown]
# ## Task 4: Baseline Model
# Evaluation metrics for our Logistic Regression baseline.

# %%
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_tfidf, y_train)
baseline_preds = baseline_model.predict(X_test_tfidf)
baseline_acc = accuracy_score(y_test, baseline_preds)

print("--- Baseline Model Classification Report ---")
print(f"Accuracy Score: {baseline_acc:.4f}\n")
print(classification_report(y_test, baseline_preds, target_names=list(label_mapping.keys())))

# %% [markdown]
# ## Task 5: Sequence Model Architecture (LSTM)
# Training an sequential deep learning model utilizing an explicit embedding space layer backed by an LSTM block.

# %%
embedding_dim = 64

model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Train the sequential model
model.fit(X_train_pad, y_train, epochs=3, batch_size=32, validation_data=(X_test_pad, y_test), verbose=1)

lstm_preds_probs = model.predict(X_test_pad)
lstm_preds = np.argmax(lstm_preds_probs, axis=1)
lstm_acc = accuracy_score(y_test, lstm_preds)

print(f"\nLSTM Sequence Model Accuracy: {lstm_acc:.4f}")

# %% [markdown]
# ## File Generation Block
# This automated segment writes all supplemental deliverables required for your repository directly onto disk.

# %%
# 1. Output model_evaluation.csv
eval_df = pd.DataFrame({
    'Model': ['Baseline (TF-IDF + Logistic Regression)', 'Sequence Model (LSTM)'],
    'Accuracy': [baseline_acc, lstm_acc]
})
eval_df.to_csv('results/model_evaluation.csv', index=False)

# 2. Output sample_predictions.txt
inv_map = {v: k for k, v in label_mapping.items()}
test_messages_orig = df['customer_message'].iloc[X_test_raw.index].values

with open('results/sample_predictions.txt', 'w') as f:
    f.write("Sample Validation Predictions Report:\n")
    f.write("="*60 + "\n")
    for i in range(10):
        f.write(f"Message: {test_messages_orig[i]}\n")
        f.write(f"True Label: {inv_map[y_test.iloc[i]]}\n")
        f.write(f"Baseline Prediction: {inv_map[baseline_preds[i]]}\n")
        f.write(f"LSTM Prediction: {inv_map[lstm_preds[i]]}\n")
        f.write("-"*60 + "\n")

# 3. Output requirements.txt
with open('requirements.txt', 'w') as f:
    f.write("pandas\nnumpy\nscikit-learn\nnltk\ntensorflow\nmatplotlib\n")

# 4. Output README.md containing Task 3 & Task 6 reflections
readme_content = """# Part 3: NLP and Sequence Modeling Mini Project

## Repository Contents
- `notebook.ipynb`: Core workbook executing full modeling runs.
- `requirements.txt`: Package dependency files for configuration.
- `results/`:
  - `model_evaluation.csv`: Accuracy matrices charting performance.
  - `sample_predictions.txt`: True vs predicted label samples.

## Explanations and Critical Thinking

### Text Vectorization Rationale (Task 3)
Computer layers and neural architectures function purely on linear algebra vectors, dot products, and multi-dimensional matrices. They cannot extract programmatic value from structural syntax or raw string characters directly. Transforming string tokens into vector tokens encodes word distribution footprints (like frequencies inside TF-IDF mapping) or rich, continuous meanings (like Dense Word Embeddings), converting language into actionable inputs.

### Attention and Transformer Reflections (Task 6)
1. **RNN Limitations**: Recurrent neural networks cycle hidden tensors timestamp-by-timestamp sequentially. Over extended sentences, backpropagating down repeated matrix functions forces numerical properties to collapse toward zero or expand infinitely, obscuring long-distance token relationships.
2. **LSTM Improvements**: LSTMs bypass gradient decay using an internal pipeline called a 'Cell State.' By filtering historical information through Input, Forget, and Output computational gates, LSTMs retain contextual text sequences across long spans.
3. **The Role of Attention**: Standard encoder-decoder frameworks bundle all semantic details into a single fixed vector block. Attention eliminates this data bottleneck by allowing decoders to dynamically scan all encoder steps, retrieving key tokens regardless of distance.
4. **Modern Transformers & Generative AI**: Transformers completely remove sequential structures in favor of parallelized Self-Attention layers. This allows training systems to process entire documents at once, boosting execution speeds on modern hardware and building the structural foundation for modern Large Language Models.
"""

with open('README.md', 'w') as f:
    f.write(readme_content)

print("All file structures successfully written onto your disk space!")